In [1]:
import re
import numpy as np
import pandas as pd

print("Libraries imported successfully! ✅")


Libraries imported successfully! ✅


In [2]:
df = pd.read_csv(
    "../data/processed/manali_hybrid_scores.csv"
)

print("Rows:", len(df))
print("Columns:", len(df.columns))
df[["name", "category", "travel_tags", "rating", "reviews"]].head(10)


Rows: 20
Columns: 29


,name,category,travel_tags,rating,reviews
0,Hadimba Devi Temple,Tourist attraction,"culture, history, nature, photography, religio...",4.6,49688
1,Old Manali snow point,Tourist attraction,"culture, family, history, nature, photography",4.6,428
2,Nehru Kund,Tourist attraction,"history, photography",4.4,7767
3,Kullu Manali River rafting,Tourist attraction,NaN,4.5,88
4,Jogini Falls,Tourist attraction,"family, history, nature, photography",4.6,10842
5,Van Vihar National Park,Tourist attraction,"family, history, nature, photography",4.2,9050
6,Manali View Point,Tourist attraction,photography,4.6,87
7,Rahala Waterfalls,Tourist attraction,"family, history, nature",4.5,797
8,Lama Dugh Trek Start Point,Tourist attraction,nature,4.6,297
9,Atal Bihari statue,Tourist attraction,history,4.5,74


In [3]:
df["travel_tags"] = (
    df["travel_tags"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

df["name_clean"] = (
    df["name"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

df[["name", "travel_tags"]].head(10)


,name,travel_tags
0,Hadimba Devi Temple,"culture, history, nature, photography, religio..."
1,Old Manali snow point,"culture, family, history, nature, photography"
2,Nehru Kund,"history, photography"
3,Kullu Manali River rafting,
4,Jogini Falls,"family, history, nature, photography"
5,Van Vihar National Park,"family, history, nature, photography"
6,Manali View Point,photography
7,Rahala Waterfalls,"family, history, nature"
8,Lama Dugh Trek Start Point,nature
9,Atal Bihari statue,history


In [4]:
def contains_any(text, keywords):
    text = str(text).lower()
    return int(any(keyword in text for keyword in keywords))


In [5]:
def infer_activity_type(row):
    text = f"{row['name_clean']} {row['travel_tags']}"

    if "waterfall" in text or "falls" in text:
        return "waterfall"

    if "temple" in text:
        return "temple"

    if "trek" in text:
        return "trekking"

    if "rafting" in text:
        return "rafting"

    if "view point" in text or "viewpoint" in text:
        return "viewpoint"

    if "bazaar" in text or "shopping" in text:
        return "shopping"

    if "park" in text or "forest" in text:
        return "nature"

    if "igloo" in text:
        return "winter_experience"

    if "valley" in text:
        return "nature"

    if "snow" in text:
        return "winter_experience"

    return "sightseeing"


df["activity_type"] = df.apply(
    infer_activity_type,
    axis=1
)

df[[
    "name",
    "activity_type"
]]


,name,activity_type
0,Hadimba Devi Temple,temple
1,Old Manali snow point,winter_experience
2,Nehru Kund,sightseeing
3,Kullu Manali River rafting,rafting
4,Jogini Falls,waterfall
5,Van Vihar National Park,nature
6,Manali View Point,viewpoint
7,Rahala Waterfalls,waterfall
8,Lama Dugh Trek Start Point,trekking
9,Atal Bihari statue,sightseeing


In [6]:
def infer_travel_style(row):
    text = f"{row['name_clean']} {row['travel_tags']}"

    styles = []

    if any(x in text for x in ["nature", "waterfall", "forest", "valley"]):
        styles.append("nature")

    if any(x in text for x in ["trek", "rafting", "snow", "adventure"]):
        styles.append("adventure")

    if any(x in text for x in ["temple", "history", "culture"]):
        styles.append("culture")

    if "photography" in text or "viewpoint" in text or "view point" in text:
        styles.append("photography")

    if "family" in text:
        styles.append("family")

    if "shopping" in text or "bazaar" in text:
        styles.append("shopping")

    if not styles:
        styles.append("general")

    return ", ".join(sorted(set(styles)))


df["travel_style"] = df.apply(
    infer_travel_style,
    axis=1
)

df[[
    "name",
    "travel_style"
]]


,name,travel_style
0,Hadimba Devi Temple,"culture, nature, photography, shopping"
1,Old Manali snow point,"adventure, culture, family, nature, photography"
2,Nehru Kund,"culture, photography"
3,Kullu Manali River rafting,adventure
4,Jogini Falls,"culture, family, nature, photography"
5,Van Vihar National Park,"culture, family, nature, photography"
6,Manali View Point,photography
7,Rahala Waterfalls,"culture, family, nature"
8,Lama Dugh Trek Start Point,"adventure, nature"
9,Atal Bihari statue,culture


In [7]:
def estimate_visit_minutes(row):
    activity = row["activity_type"]

    duration_map = {
        "temple": 60,
        "waterfall": 90,
        "trekking": 150,
        "rafting": 120,
        "viewpoint": 45,
        "shopping": 90,
        "nature": 120,
        "winter_experience": 90,
        "sightseeing": 60
    }

    return duration_map.get(activity, 60)


df["estimated_visit_minutes"] = df.apply(
    estimate_visit_minutes,
    axis=1
)

df[[
    "name",
    "activity_type",
    "estimated_visit_minutes"
]]


,name,activity_type,estimated_visit_minutes
0,Hadimba Devi Temple,temple,60
1,Old Manali snow point,winter_experience,90
2,Nehru Kund,sightseeing,60
3,Kullu Manali River rafting,rafting,120
4,Jogini Falls,waterfall,90
5,Van Vihar National Park,nature,120
6,Manali View Point,viewpoint,45
7,Rahala Waterfalls,waterfall,90
8,Lama Dugh Trek Start Point,trekking,150
9,Atal Bihari statue,sightseeing,60


In [8]:
def infer_price_level(row):
    text = f"{row['name_clean']} {row['travel_tags']}"

    if "rafting" in text:
        return 3

    if "igloo" in text:
        return 3

    if "shopping" in text or "bazaar" in text:
        return 2

    # Many outdoor/public attractions have relatively low direct entry cost.
    return 1


df["estimated_price_level"] = df.apply(
    infer_price_level,
    axis=1
)

df[[
    "name",
    "activity_type",
    "estimated_price_level"
]]


,name,activity_type,estimated_price_level
0,Hadimba Devi Temple,temple,2
1,Old Manali snow point,winter_experience,1
2,Nehru Kund,sightseeing,1
3,Kullu Manali River rafting,rafting,3
4,Jogini Falls,waterfall,1
5,Van Vihar National Park,nature,1
6,Manali View Point,viewpoint,1
7,Rahala Waterfalls,waterfall,1
8,Lama Dugh Trek Start Point,trekking,1
9,Atal Bihari statue,sightseeing,1


In [9]:
df["is_outdoor"] = (
    df["activity_type"]
    .isin([
        "waterfall",
        "trekking",
        "viewpoint",
        "nature",
        "winter_experience",
        "rafting"
    ])
    .astype(int)
)

df["is_family_friendly"] = (
    df["family"].fillna(0).astype(float) > 0
).astype(int)

df["is_photography_friendly"] = (
    df["photography"].fillna(0).astype(float) > 0
).astype(int)

df["is_adventure"] = (
    df["activity_type"]
    .isin([
        "trekking",
        "rafting",
        "winter_experience"
    ])
    .astype(int)
)

df[[
    "name",
    "is_outdoor",
    "is_family_friendly",
    "is_photography_friendly",
    "is_adventure"
]].head(20)


,name,is_outdoor,is_family_friendly,is_photography_friendly,is_adventure
0,Hadimba Devi Temple,0,0,1,0
1,Old Manali snow point,1,1,1,1
2,Nehru Kund,0,0,1,0
3,Kullu Manali River rafting,1,0,0,1
4,Jogini Falls,1,1,1,0
5,Van Vihar National Park,1,1,1,0
6,Manali View Point,1,0,1,0
7,Rahala Waterfalls,1,1,0,0
8,Lama Dugh Trek Start Point,1,0,0,1
9,Atal Bihari statue,0,0,0,0


In [10]:
def infer_best_time(row):
    activity = row["activity_type"]

    if activity in ["viewpoint", "photography"]:
        return "morning_or_evening"

    if activity in ["trekking", "nature", "waterfall"]:
        return "morning"

    if activity == "shopping":
        return "afternoon_or_evening"

    if activity == "temple":
        return "morning"

    return "daytime"


df["estimated_best_time"] = df.apply(
    infer_best_time,
    axis=1
)

df[[
    "name",
    "activity_type",
    "estimated_best_time"
]]


,name,activity_type,estimated_best_time
0,Hadimba Devi Temple,temple,morning
1,Old Manali snow point,winter_experience,daytime
2,Nehru Kund,sightseeing,daytime
3,Kullu Manali River rafting,rafting,daytime
4,Jogini Falls,waterfall,morning
5,Van Vihar National Park,nature,morning
6,Manali View Point,viewpoint,morning_or_evening
7,Rahala Waterfalls,waterfall,morning
8,Lama Dugh Trek Start Point,trekking,morning
9,Atal Bihari statue,sightseeing,daytime


In [11]:
def build_planning_description(row):
    parts = []

    parts.append(
        f"{row['name']} is a {row['activity_type']} experience in Manali."
    )

    parts.append(
        f"Suitable travel styles: {row['travel_style']}."
    )

    parts.append(
        f"Estimated visit time: {int(row['estimated_visit_minutes'])} minutes."
    )

    if row["is_photography_friendly"]:
        parts.append("Useful for photography.")

    if row["is_family_friendly"]:
        parts.append("Has a family-friendly signal.")

    if row["is_adventure"]:
        parts.append("Has an adventure-oriented signal.")

    return " ".join(parts)


df["planning_description"] = df.apply(
    build_planning_description,
    axis=1
)

df[[
    "name",
    "planning_description"
]].head(10)


,name,planning_description
0,Hadimba Devi Temple,Hadimba Devi Temple is a temple experience in ...
1,Old Manali snow point,Old Manali snow point is a winter_experience e...
2,Nehru Kund,Nehru Kund is a sightseeing experience in Mana...
3,Kullu Manali River rafting,Kullu Manali River rafting is a rafting experi...
4,Jogini Falls,Jogini Falls is a waterfall experience in Mana...
5,Van Vihar National Park,Van Vihar National Park is a nature experience...
6,Manali View Point,Manali View Point is a viewpoint experience in...
7,Rahala Waterfalls,Rahala Waterfalls is a waterfall experience in...
8,Lama Dugh Trek Start Point,Lama Dugh Trek Start Point is a trekking exper...
9,Atal Bihari statue,Atal Bihari statue is a sightseeing experience...


In [12]:
planning_columns = [
    "name",
    "latitude",
    "longitude",
    "rating",
    "reviews",
    "category",
    "travel_tags",
    "activity_type",
    "travel_style",
    "estimated_visit_minutes",
    "estimated_price_level",
    "estimated_best_time",
    "planning_description"
]

available_columns = [
    col for col in planning_columns
    if col in df.columns
]

df["data_completeness"] = (
    df[available_columns]
    .notna()
    .sum(axis=1)
    / len(available_columns)
)

df[[
    "name",
    "data_completeness"
]].sort_values(
    "data_completeness"
).head(10)


,name,data_completeness
0,Hadimba Devi Temple,1.0
1,Old Manali snow point,1.0
2,Nehru Kund,1.0
3,Kullu Manali River rafting,1.0
4,Jogini Falls,1.0
5,Van Vihar National Park,1.0
6,Manali View Point,1.0
7,Rahala Waterfalls,1.0
8,Lama Dugh Trek Start Point,1.0
9,Atal Bihari statue,1.0


In [13]:
enriched_columns = [
    "name",
    "activity_type",
    "travel_style",
    "estimated_visit_minutes",
    "estimated_price_level",
    "estimated_best_time",
    "is_outdoor",
    "is_family_friendly",
    "is_photography_friendly",
    "is_adventure",
    "planning_description",
    "data_completeness"
]

df[enriched_columns]


,name,activity_type,travel_style,estimated_visit_minutes,estimated_price_level,estimated_best_time,is_outdoor,is_family_friendly,is_photography_friendly,is_adventure,planning_description,data_completeness
0,Hadimba Devi Temple,temple,"culture, nature, photography, shopping",60,2,morning,0,0,1,0,Hadimba Devi Temple is a temple experience in ...,1.0
1,Old Manali snow point,winter_experience,"adventure, culture, family, nature, photography",90,1,daytime,1,1,1,1,Old Manali snow point is a winter_experience e...,1.0
2,Nehru Kund,sightseeing,"culture, photography",60,1,daytime,0,0,1,0,Nehru Kund is a sightseeing experience in Mana...,1.0
3,Kullu Manali River rafting,rafting,adventure,120,3,daytime,1,0,0,1,Kullu Manali River rafting is a rafting experi...,1.0
4,Jogini Falls,waterfall,"culture, family, nature, photography",90,1,morning,1,1,1,0,Jogini Falls is a waterfall experience in Mana...,1.0
5,Van Vihar National Park,nature,"culture, family, nature, photography",120,1,morning,1,1,1,0,Van Vihar National Park is a nature experience...,1.0
6,Manali View Point,viewpoint,photography,45,1,morning_or_evening,1,0,1,0,Manali View Point is a viewpoint experience in...,1.0
7,Rahala Waterfalls,waterfall,"culture, family, nature",90,1,morning,1,1,0,0,Rahala Waterfalls is a waterfall experience in...,1.0
8,Lama Dugh Trek Start Point,trekking,"adventure, nature",150,1,morning,1,0,0,1,Lama Dugh Trek Start Point is a trekking exper...,1.0
9,Atal Bihari statue,sightseeing,culture,60,1,daytime,0,0,0,0,Atal Bihari statue is a sightseeing experience...,1.0


In [14]:
low_information = df[
    (df["activity_type"] == "sightseeing")
    | (df["travel_style"] == "general")
]

low_information[[
    "name",
    "activity_type",
    "travel_style",
    "travel_tags"
]]


,name,activity_type,travel_style,travel_tags
2,Nehru Kund,sightseeing,"culture, photography","history, photography"
9,Atal Bihari statue,sightseeing,culture,history
11,Mini Switzerland Manali,sightseeing,family,family
13,Himachal PARDESH,sightseeing,general,
16,Himalayan Igloo,winter_experience,general,


In [15]:
print("Activity types:")
print(
    df["activity_type"]
    .value_counts()
)

print("\nTravel style combinations:")
print(
    df["travel_style"]
    .value_counts()
)


Activity types:
activity_type
sightseeing          4
waterfall            3
nature               3
viewpoint            3
temple               2
winter_experience    2
rafting              1
trekking             1
shopping             1
Name: count, dtype: int64

Travel style combinations:
travel_style
culture, family, nature, photography               3
culture                                            2
culture, nature                                    2
general                                            2
culture, nature, photography, shopping             1
adventure, culture, family, nature, photography    1
culture, photography                               1
adventure                                          1
photography                                        1
culture, family, nature                            1
adventure, nature                                  1
family                                             1
culture, family, photography                       1
nature 

In [16]:
output_path = "../data/processed/manali_travel_enriched.csv"

df.to_csv(
    output_path,
    index=False
)

print(f"✅ Saved: {output_path}")
print("Final shape:", df.shape)


✅ Saved: ../data/processed/manali_travel_enriched.csv
Final shape: (20, 41)
